# 01 - Fundamentos de Árboles de Decisión (CART), Gini y Entropía

## Curso de Machine Learning: Modelos de Clasificación

Los **Árboles de Decisión (Decision Trees)** son modelos supervisados fundamentales tanto por su capacidad para modelar relaciones no lineales e interacciones complejas entre características, como por su alta **interpretabilidad**. Además, constituyen el bloque constructivo esencial de las arquitecturas de ensamble más exitosas de la ciencia de datos moderna (Random Forest, Gradient Boosting, XGBoost y LightGBM).

---

### Contenidos
1. Anatomía Estructural de un Árbol de Decisión
2. Criterios de Impureza: Índice Gini, Entropía de Shannon y Error de Clasificación
3. Comparativa Gráfica de Medidas de Impureza
4. Algoritmo CART y Ganancia de Información (Reducción de Impureza)
5. Ajuste Práctico de `DecisionTreeClassifier` en Scikit-Learn
6. Visualización Gráfica e Interpretación Estructural (`plot_tree`, `export_text`)
7. Conclusiones


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.datasets import load_breast_cancer, load_iris
from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeClassifier, plot_tree, export_text
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix, ConfusionMatrixDisplay

plt.style.use('seaborn-v0_8-whitegrid' if 'seaborn-v0_8-whitegrid' in plt.style.available else 'default')
np.random.seed(42)
print("Bibliotecas cargadas y entorno listo.")


## 1. Anatomía Estructural de un Árbol de Decisión

Un árbol de clasificación segmenta jerárquicamente el espacio de características $\mathcal{X} \subset \mathbb{R}^D$ en rectángulos o hipercubos disjuntos ortogonales a los ejes:

- **Nodo Raíz (Root Node)**: Contiene la totalidad de los datos iniciales y define la primera regla de bifurcación condicional.
- **Nodos Internos de Decisión**: Evalúan una condición de umbral $x_j \le \theta$.
- **Hojas o Nodos Terminales (Leaf Nodes)**: Asignan una etiqueta de clase definitiva basada en la moda o las probabilidades relativas de las muestras que cayeron en ese subconjunto.


## 2. Criterios de Impureza: Gini, Entropía y Error

Dado un nodo $t$ con proporciones empíricas de clase $p(i \mid t) = \frac{N_{t, i}}{N_t}$ para $i = 1, \dots, C$:

### 1. Índice de Impureza Gini
Mide la probabilidad de clasificar erróneamente un elemento elegido al azar si se etiquetara aleatoriamente según la distribución de clases del nodo:
$$I_G(t) = 1 - \sum_{i=1}^C p(i \mid t)^2$$
- Nodo puro ($p=1$ para una clase): $I_G = 0$.
- Máxima impureza (caso binario equilibrado $p_1=p_2=0.5$): $I_G = 0.5$.

### 2. Entropía de Shannon (Ganancia de Información)
Derivada de la teoría de la información de Claude Shannon, mide el desorden o incertidumbre promedio del nodo:
$$H(t) = -\sum_{i=1}^C p(i \mid t) \log_2(p(i \mid t))$$
- Nodo puro: $H = 0$.
- Máxima impureza (caso binario equilibrado): $H = 1.0$ bit.

### 3. Error de Clasificación (Misclassification Rate)
$$E(t) = 1 - \max_i p(i \mid t)$$


In [ ]:
# Comparacion Grafica de las Medidas de Impureza en Clasificacion Binaria
p = np.linspace(0.0001, 0.9999, 500)

gini = 2 * p * (1 - p)                     # Normalizado para comparacion (Gini * 2)
entropy = - (p * np.log2(p) + (1 - p) * np.log2(1 - p))
misclass = 1 - np.maximum(p, 1 - p)

plt.figure(figsize=(10, 5))
plt.plot(p, entropy, label='Entropia de Shannon (bits)', color='#2b5c8f', lw=2.5)
plt.plot(p, gini, label='Indice Gini (Escalado x2)', color='#d95f02', lw=2, linestyle='--')
plt.plot(p, misclass * 2, label='Error de Clasificacion (Escalado x2)', color='#2ca02c', lw=2, linestyle=':')

plt.axvline(0.5, color='gray', linestyle='--', alpha=0.6)
plt.xlabel('Probabilidad de la Clase 1 ($p$)', fontsize=11, fontweight='bold')
plt.ylabel('Medida de Impureza', fontsize=11, fontweight='bold')
plt.title('Comparativa de Criterios de Impureza para Clasificacion Binaria', fontsize=13, fontweight='bold')
plt.legend(fontsize=11)
plt.tight_layout()
plt.show()


## 3. Algoritmo CART y Reducción de Impureza

El algoritmo **CART (Classification and Regression Trees)** implementa divisiones estrictamente binarias. Para cada nodo padre $t$, busca de forma exhaustiva la característica $j$ y el umbral $\theta$ que maximicen la **Reducción de Impureza (Ganancia de Información)**:

$$\Delta I(s, t) = I(t) - \left( \frac{N_{izq}}{N_t} I(t_{izq}) + \frac{N_{der}}{N_t} I(t_{der}) \right)$$

El algoritmo procede de forma golosa (*greedy*) de arriba hacia abajo hasta que se alcanza un criterio de parada (pureza total, profundidad máxima o mínimo de muestras).


In [ ]:
# Entrenamiento de un Arbol en el Dataset Iris para Visualizacion
iris = load_iris()
X_iris, y_iris = iris.data, iris.target

# Usamos solo dos caracteristicas para visualizacion clara
X_train_iris, X_test_iris, y_train_iris, y_test_iris = train_test_split(
    X_iris, y_iris, test_size=0.3, random_state=42, stratify=y_iris
)

tree_clf = DecisionTreeClassifier(criterion='gini', max_depth=3, random_state=42)
tree_clf.fit(X_train_iris, y_train_iris)

y_pred_iris = tree_clf.predict(X_test_iris)
print(f"Exactitud en prueba (max_depth=3): {accuracy_score(y_test_iris, y_pred_iris)*100:.2f}%
")


In [ ]:
# Visualizacion Estructural Completa con plot_tree
plt.figure(figsize=(16, 9))
plot_tree(
    tree_clf, 
    feature_names=iris.feature_names, 
    class_names=iris.target_names, 
    filled=True, 
    rounded=True, 
    fontsize=10
)
plt.title('Arbol de Decision Entrenado en Dataset Iris (max_depth=3)', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()


In [ ]:
# Exportacion en formato textual interpretable
reglas_texto = export_text(tree_clf, feature_names=iris.feature_names)
print("Reglas de Decision Logicas Extraidas del Arbol:
")
print(reglas_texto)


## 4. Conclusiones y Próximos Pasos

1. **Interpretabilidad Nativa**: Los árboles permiten rastrear el camino exacto de decisiones mediante condiciones lógicas simples tipo *IF-THEN*.
2. **Invarianza a Escalas**: A diferencia de KNN o SVM, los árboles de decisión son completamente invariantes al escalado monótono de variables (`StandardScaler` no es necesario).
3. **Gini vs. Entropía**: En la práctica, Gini y Entropía producen árboles casi idénticos en el 98% de los casos. Gini es computacionalmente más rápido porque no requiere calcular logaritmos.
4. **Riesgo Crítico**: Los árboles sin restricciones crecen hasta memorizar cada muestra individual, generando sobreajuste severo.

En el siguiente cuaderno, **02_Hiperparametros_Poda_y_Visualizacion.ipynb**, abordaremos las técnicas de pre-poda y post-poda por costo-complejidad (`ccp_alpha`) para controlar rigurosamente el sobreajuste.
